In [ ]:
!pip install pysentimiento transformers torch scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.3 MB/s eta 0:00:00


In [ ]:
from torch.optim import AdamW
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from pysentimiento.preprocessing import preprocess_tweet

In [ ]:

# =========================================================
# SEED
# =========================================================

def set_seed(seed=42):

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


set_seed(42)

In [ ]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "pysentimiento/robertuito-base-cased"

MAX_LEN = 120

BATCH_SIZE = 16

EPOCHS = 15

LR = 2e-5

CAPA_1 = 2048

CAPA_2 = 1024

PATIENCE = 10

WEIGHT_DECAY = 0.001278

DP1 = 0.153572

DP2 = 0.136504

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Usando dispositivo: {device}")

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


Usando dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
# =========================================================
# DATASET
# =========================================================

class HumorTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len
    ):

        self.texts = texts

        self.labels = labels

        self.tokenizer = tokenizer

        self.max_len = max_len

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, item):

        text = str(self.texts[item])

        text = preprocess_tweet(
            text,
            lang="es"
        )

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            return_attention_mask=True
        )

        item_dict = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"]
        }

        if self.labels is not None:

            item_dict["labels"] = int(
                self.labels[item]
            )

        return item_dict

In [ ]:

# =========================================================
# MODELO
# =========================================================

class HumorRoBERTuito(nn.Module):

    def __init__(self):

        super().__init__()

        print("Cargando RoBERTuito...")

        self.roberta = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # =============================================
        # Congelar primeras capas
        # =============================================

        for name, param in self.roberta.named_parameters():

            if "encoder.layer.0" in name:
                param.requires_grad = False

            if "encoder.layer.1" in name:
                param.requires_grad = False

            if "encoder.layer.2" in name:
                param.requires_grad = False

        self.input_dim = 768 * 2

        self.classifier = nn.Sequential(

            nn.Linear(
                self.input_dim,
                CAPA_1
            ),

            nn.LayerNorm(CAPA_1),

            nn.GELU(),

            nn.Dropout(DP1),

            nn.Linear(
                CAPA_1,
                CAPA_2
            ),

            nn.LayerNorm(CAPA_2),

            nn.GELU(),

            nn.Dropout(DP2),

            nn.Linear(
                CAPA_2,
                2
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_state = outputs.last_hidden_state

        # =============================================
        # MASK
        # =============================================

        mask_expanded = (
            attention_mask
            .unsqueeze(-1)
            .expand(hidden_state.size())
            .float()
        )

        # =============================================
        # CLS TOKEN
        # =============================================

        cls_token = hidden_state[:, 0]

        # =============================================
        # MEAN POOLING
        # =============================================

        sum_embeddings = torch.sum(
            hidden_state * mask_expanded,
            dim=1
        )

        sum_mask = torch.clamp(
            mask_expanded.sum(dim=1),
            min=1e-9
        )

        mean_pooled = sum_embeddings / sum_mask

        # =============================================
        # CONCAT
        # =============================================

        concat_vector = torch.cat(
            (cls_token, mean_pooled),
            dim=1
        )

        logits = self.classifier(
            concat_vector
        )

        return logits

In [ ]:

# =========================================================
# CARGAR DATOS
# =========================================================

print("Leyendo dataset...")

df = pd.read_json(
    "./Datasets/dataset_humor_train.json",
    lines=True
)

textos = df["text"].values

etiquetas = df["klass"].values

# =========================================================
# SPLIT
# =========================================================

X_train, X_val, y_train, y_val = train_test_split(
    textos,
    etiquetas,
    test_size=0.10,
    stratify=etiquetas,
    random_state=42
)

# =========================================================
# DATASETS
# =========================================================

train_dataset = HumorTextDataset(
    X_train,
    y_train,
    tokenizer,
    MAX_LEN
)

val_dataset = HumorTextDataset(
    X_val,
    y_val,
    tokenizer,
    MAX_LEN
)

# =========================================================
# DYNAMIC PADDING
# =========================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# =========================================================
# DATALOADERS
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=True
)

# =========================================================
# CLASS WEIGHTS
# =========================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(etiquetas),
    y=etiquetas
)

weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)


Leyendo dataset...


In [ ]:
unique, counts = np.unique(
    etiquetas,
    return_counts=True
)

print(
    "Conteo de clases:",
    dict(zip(unique, counts))
)


Conteo de clases: {np.int64(0): np.int64(6588), np.int64(1): np.int64(3812)}


In [ ]:

# =========================================================
# MODELO
# =========================================================

model = HumorRoBERTuito()

model = model.to(device)

# =========================================================
# OPTIMIZER
# =========================================================

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# =========================================================
# SCHEDULER
# =========================================================

total_steps = len(train_loader) * EPOCHS

warmup_steps = int(
    total_steps * 0.1
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# =========================================================
# LOSS
# =========================================================

loss_fn = nn.CrossEntropyLoss(
    weight=weights
)

# =========================================================
# AMP
# =========================================================

scaler = torch.cuda.amp.GradScaler()

# =========================================================
# TRAINING
# =========================================================

best_f1 = 0.0

counter = 0

best_model_path = "mejor_modelo.pth"

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    print("-" * 30)

    # =====================================================
    # TRAIN
    # =====================================================

    model.train()

    train_losses = []

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        targets = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(
                outputs,
                targets
            )

        train_losses.append(
            loss.item()
        )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)

        scaler.update()

        scheduler.step()

    print(
        f"Train Loss: {np.mean(train_losses):.4f}"
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    model.eval()

    predictions = []

    real_values = []

    val_losses = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            targets = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(
                outputs,
                targets
            )

            val_losses.append(
                loss.item()
            )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            real_values.extend(
                targets.cpu().numpy()
            )

    val_f1 = f1_score(
        real_values,
        predictions,
        average="macro"
    )

    print(
        f"Val Loss: {np.mean(val_losses):.4f}"
    )

    print(
        f"Val F1 Macro: {val_f1:.4f}"
    )

    # =====================================================
    # SAVE BEST
    # =====================================================

    if val_f1 > best_f1:

        best_f1 = val_f1

        counter = 0

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Nuevo mejor modelo guardado.")

    else:

        counter += 1

        print(
            f"Sin mejora ({counter}/{PATIENCE})"
        )

    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if counter >= PATIENCE:

        print("\nEarly Stopping")

        break

# =========================================================
# LOAD BEST MODEL
# =========================================================

model.load_state_dict(
    torch.load(best_model_path)
)

print(
    f"\nMejor F1 Macro: {best_f1:.4f}"
)



Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

/tmp/ipykernel_1598/3121834621.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1598/3121834621.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/15
------------------------------
Train Loss: 0.4412
Val Loss: 0.3595
Val F1 Macro: 0.8496
Nuevo mejor modelo guardado.

Epoch 2/15
------------------------------


/tmp/ipykernel_1598/3121834621.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

Train Loss: 0.0022
Val Loss: 1.2586
Val F1 Macro: 0.8635
Sin mejora (8/10)

Epoch 13/15
------------------------------


/tmp/ipykernel_1598/3121834621.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Train Loss: 0.0005
Val Loss: 1.2773
Val F1 Macro: 0.8669
Sin mejora (9/10)

Epoch 14/15
------------------------------


/tmp/ipykernel_1598/3121834621.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Train Loss: 0.0005
Val Loss: 1.2541
Val F1 Macro: 0.8640
Sin mejora (10/10)

Early Stopping

Mejor F1 Macro: 0.8715


In [ ]:
# =========================================================
# TEST
# =========================================================

print("\nCargando dataset de prueba...")

dataset_test = pd.read_json(
    "./Datasets/dataset_humor_test.json",
    lines=True
)

X_test = dataset_test["text"].values

# =========================================================
# TEST DATASET
# =========================================================

test_dataset = HumorTextDataset(
    X_test,
    labels=None,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# =========================================================
# TEST DATALOADER
# =========================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

# =========================================================
# INFERENCE
# =========================================================

model.eval()

predictions_list = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        predictions_list.extend(
            preds.cpu().numpy()
        )

y_pred_final = np.array(
    predictions_list
)

# =========================================================
# RESULTADOS
# =========================================================

print("\nPredicciones generadas.")

unique, counts = np.unique(
    y_pred_final,
    return_counts=True
)

print(
    "Conteo de clases:",
    dict(zip(unique, counts))
)

print(
    "\nPrimeras predicciones:"
)

print(
    y_pred_final[:20]
)


Cargando dataset de prueba...

Predicciones generadas.
Conteo de clases: {np.int64(0): np.int64(3454), np.int64(1): np.int64(2146)}

Primeras predicciones:
[0 0 0 1 0 1 1 0 0 0 0 1 1 1 0 0 0 1 1 0]


In [ ]:
import torch.nn.functional as F  # ← Necesitamos esto para Softmax

# =========================================================
# INFERENCE WITH PROBABILITIES
# =========================================================

model.eval()

# Crearemos una lista para guardar las probabilidades de la clase positiva (Humor = 1)
probabilidades_humor = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 1. Si estás usando AutoModelForSequenceClassification de HF, los logits vienen dentro de outputs.logits
        # Si tu modelo mapeado devuelve los logits directo, déjalo solo como 'outputs'
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # 2. Aplicamos Softmax para convertir logits en probabilidades (0.0 a 1.0)
        probs = F.softmax(logits, dim=1)

        # 3. Extraemos la probabilidad de la Clase 1 (columna 1: "Sí es humor")
        # .cpu().numpy() lo pasa a la memoria RAM normal
        prob_clase_1 = probs[:, 1].cpu().numpy()

        # 4. Acumulamos en nuestra lista
        probabilidades_humor.extend(prob_clase_1)

# Convertimos a un array final de NumPy
y_prob_robertuito = np.array(probabilidades_humor)

# =========================================================
# RESULTADOS
# =========================================================
print("\nProbabilidades generadas con Robertuito.")
print("Primeras 5 probabilidades de que sea humor (Clase 1):")
print(y_prob_robertuito[:20])


Probabilidades generadas con Robertuito.
Primeras 5 probabilidades de que sea humor (Clase 1):
[0.98122716 0.00628709 0.00718535 0.02080709 0.00613684 0.01160623
 0.07340738 0.0176546  0.0054697  0.9604674  0.440412   0.9836827
 0.34272602 0.008206   0.9562439  0.00572872 0.00639209 0.00568687
 0.00609024 0.00757319]


In [ ]:
print(y_prob_robertuito[:20])

[0.98122716 0.00628709 0.00718535 0.02080709 0.00613684 0.01160623
 0.07340738 0.0176546  0.0054697  0.9604674  0.440412   0.9836827
 0.34272602 0.008206   0.9562439  0.00572872 0.00639209 0.00568687
 0.00609024 0.00757319]


In [ ]:
def guardar_resultados(datos, archivo):
    df = pd.DataFrame(datos, columns=['klass'])
    df['id'] = df.index + 1
    df = df[['id', 'klass']]

    df.to_csv(archivo, index=False)
    print(f"Archivo guardado exitosamente: {archivo}")

In [ ]:
nombre_archivo = f"robertuito_predicciones_V12.csv"
guardar_resultados(y_pred_final, nombre_archivo)

Archivo guardado exitosamente: robertuito_predicciones_V12.csv


In [ ]:
nombre_archivo = f"robertuito_probs_{LR}_{CAPA_1}_{CAPA_2}_{WEIGHT_DECAY}_{BATCH_SIZE}_{DP1}_{DP2}_val.csv"
guardar_resultados(y_prob_robertuito, nombre_archivo)

Archivo guardado exitosamente: robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_val.csv


In [ ]:
import pandas as pd

pd_LLM = pd.read_csv("LLM_probs_umbral_0.89_app.csv")
probs_LLM = np.array(pd_LLM['klass'])
probs_LLM

array([8.67035747e-01, 4.07013809e-03, 2.47262325e-03, ...,
       7.31058538e-01, 8.04085925e-04, 9.92422760e-01])

In [ ]:

pd_LLM_1 = pd.read_csv("LLM_8791_validacion.csv")
probs_LLM_1 = np.array(pd_LLM_1['klass'])
probs_LLM_1

array([8.67187500e-01, 4.60815430e-03, 2.18200684e-03, ...,
       7.30468750e-01, 8.04901123e-04, 9.92187500e-01])

In [ ]:
pd_betito = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_val.csv")
probs_betito = np.array(pd_betito['klass'])
probs_betito

array([0.98122716, 0.00628709, 0.00718535, ..., 0.2970919 , 0.00625764,
       0.99288225])

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# p_roberta = probabilidades de clase 1 que guardaste de RoBERTuito (array de floats 0-1)
# test_scores = los scores que ya tienes de Gemma V2

p_gemma   = probs_LLM
p_gemma_1   = probs_LLM_1
p_roberta = probs_betito

# Búsqueda de pesos óptimos en validación
best_w, best_f1 = 0.5, 0.0
for w in np.arange(0.1, 1.0, 0.05):
    p_ens = w * p_gemma_1 + (1 - w) * p_roberta
    # también busca umbral óptimo para cada peso
    for thresh in np.arange(0.3, 0.7, 0.02):
        preds = (p_ens >= thresh).astype(int)
        f1 = f1_score(y_val, preds, average='macro')
        if f1 > best_f1:
            best_f1, best_w, best_thresh = f1, w, thresh

print(f"Mejor peso Gemma: {best_w:.2f}  Umbral: {best_thresh:.2f}  F1 val: {best_f1:.4f}")

Mejor peso Gemma: 0.70  Umbral: 0.68  F1 val: 0.8804


In [ ]:
pd_LLM_test = pd.read_csv("LLM_probs_0.54_app_test.csv")
test_LLM = np.array(pd_LLM_test['klass'])
test_LLM

array([0.24511719, 0.07568359, 0.40820312, ..., 0.9921875 , 0.26953125,
       0.98828125])

In [ ]:
pd_LLM_test_1 = pd.read_csv("LLM_8791_test.csv")
test_LLM_1 = np.array(pd_LLM_test_1['klass'])
test_LLM_1

array([0.73046875, 0.0534668 , 0.97265625, ..., 0.9921875 , 0.37695312,
       0.96875   ])

In [ ]:
pd_betito_test = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_test.csv")
test_betito = np.array(pd_betito_test['klass'])
test_betito

array([4.0985453e-03, 8.8077740e-04, 1.2839267e-03, ..., 9.9921690e-01,
       9.8251430e-04, 9.9922860e-01])

In [ ]:
import numpy as np
import pandas as pd

# Tus parámetros óptimos encontrados en val
best_w      = 0.70
best_thresh = 0.68

p_gemma_test = test_LLM
p_gemma_test_1 = test_LLM_1
p_roberta_test = test_betito

# Ensemble sobre test
p_ensemble_test = best_w * p_gemma_test_1 + (1 - best_w) * p_roberta_test

# Predicciones finales
test_preds_ensemble = (p_ensemble_test >= best_thresh).astype(int)

# Distribución (verifica que no esté colapsado)
unique, counts = np.unique(test_preds_ensemble, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

pd.DataFrame({
    'id'    : range(1, len(test_preds_ensemble) + 1),
    'klass' : test_preds_ensemble,
    #'score' : p_ensemble_test,
}).to_csv('GG_Robertuito_V5.csv', index=False)

print('✅ GG_Robertuito_V5.csv listo')

{0: 3533, 1: 2067}
✅ GG_Robertuito_V5.csv listo


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# 1. Tus arrays de probabilidades de la Clase 1 (asegúrate de que todos tengan la misma longitud)
p_gemma_old = probs_LLM      # Tu modelo Gemma anterior
p_gemma_new = probs_LLM_1    # Tu nuevo Gemma 3 optimizado
p_roberta   = probs_betito   # RoBERTuito

# Variables para guardar la mejor combinación encontrada en validación
best_weights = (0.33, 0.33, 0.34)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos y umbral...")

# 2. Grid Search de Pesos (Paso de 0.05)
for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):

        # El tercer peso se calcula automáticamente para que sumen 1.0
        w_roberta = 1.0 - w_gemma_old - w_gemma_new

        # Pequeño control por flotantes imprecisos de Python
        if w_roberta < 0.0:
            continue

        # 3. Calculamos la probabilidad combinada del ensamble para este intento
        p_ens = (w_gemma_old * p_gemma_old) + (w_gemma_new * p_gemma_new) + (w_roberta * p_roberta)

        # 4. Búsqueda del umbral óptimo de clasificación para esta combinación de pesos
        for thresh in np.arange(0.3, 0.71, 0.02):
            preds = (p_ens >= thresh).astype(int)

            # NOTA: Asegúrate de que 'y_val' corresponda exactamente al mismo orden de tus arrays de probs
            f1 = f1_score(y_val, preds, average='macro')

            # Si encontramos un F1-Score más alto, guardamos la configuración
            if f1 > best_f1:
                best_f1 = f1
                best_weights = (w_gemma_old, w_gemma_new, w_roberta)
                best_thresh = thresh

# ── RESULTADOS FINALES ───────────────────────────────────────────────────────
w_old, w_new, w_rob = best_weights
print("\n¡Búsqueda completada con éxito! 🚀")
print("─" * 50)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):   {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):   {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):     {w_rob:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob:.2f})")
print("─" * 50)
print(f"Umbral de decisión (Threshold): {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:    {best_f1:.4f}")
print("─" * 50)

Buscando la combinación óptima de pesos y umbral...

¡Búsqueda completada con éxito! 🚀
──────────────────────────────────────────────────
Pesos óptimos encontrados:
  • Gemma Anterior (p_gemma):   0.00
  • Gemma Nuevo (p_gemma_1):   0.75
  • RoBERTuito (p_roberta):     0.25
  (Suma total de pesos: 1.00)
──────────────────────────────────────────────────
Umbral de decisión (Threshold): 0.70
Mejor Macro F1-Score en Val:    0.8816
──────────────────────────────────────────────────


In [ ]:

p_gemma_test = test_LLM
p_gemma_test_1 = test_LLM_1
p_roberta_test = test_betito


p_ens_test = (0.00 * p_gemma_test) + (0.75 * p_gemma_test_1) + (0.25 * p_roberta_test)
preds_finales_test = (p_ens_test >= 0.70).astype(int)

unique, counts = np.unique(test_preds_ensemble, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv('GG_GG_Robertuito_V1.csv', index=False)

print('✅ GG_Robertuito_V5.csv listo')

{0: 3533, 1: 2067}
✅ GG_Robertuito_V5.csv listo


In [ ]:
#NeuroEvolucion
model = HumorRoBERTuito()

model = model.to(device)

model.load_state_dict(
    torch.load("mejor_modelo.pth")
)

Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

<All keys matched successfully>

In [ ]:
import torch
import random
import numpy as np
from multiprocessing import  cpu_count
from sklearn.metrics import f1_score
from joblib import Parallel, delayed


#------------------------------------------------------------------------
#------------------------------------------------------------------------
# Funciones del algoritmo genético
#------------------------------------------------------------------------
#------------------------------------------------------------------------


_TAM_ENTRADA = X_train.shape[0]
_TAM_SALIDA = 2 # 3 clases

def copiar_red_neuronal(red_neuronal, prob_global=0.8, prob_alterar=0.5, intensidad_ruido=0.01, excluir_bias=True):
    """
    Genera variantes del modelo preentrenado "red_neuronal" de manera aleatoria y con una constante de intensidad del ruido
    para controlar el ruido por añadir a los parámetros

    red_neuronal: modelo de la red neuronal preentrenada
    prob_global: indica la probabilidad mínima para realizar una versión modificada de la red neuronal preentrenada
    prob_alterar: indica la probabilidad de modificar el parámetro (pesos y/o bias)
    intensidad_ruido: constante que modifica la intensidad de ruido que se agrega a los pesos y/o bias para generar una variante de la red neuronal preentrenada
    excluir_bias: indica si se altera el bias o no.

    """
    red_destino = HumorRoBERTuito()
    red_destino.load_state_dict(red_neuronal.state_dict())

    if random.random() > prob_global:
        # No hacer cambios, devolver copia exacta
        return red_destino

    with torch.no_grad():
        for name, param in red_destino.named_parameters():
            # Saltar bias si está configurado
            if excluir_bias and 'bias' in name:
                continue

            # Decidir si alteramos este parámetro específico
            if random.random() < prob_alterar:
                # Devuelve un tensor con el mismo tamaño que la entrada con números aleatorios
                # de una distribución normal con media 0 y varianza 1.
                ruido = torch.randn_like(param) * intensidad_ruido
                param.add_(ruido)
    return red_destino

#------------------------------------------------------------------------
def inicializar_poblacion(red_neuronal, tam_poblacion):
    """
    Crea la población con versiones modificadas de la red neuronal preentrenada
    """
    poblacion = []
    for _ in range(tam_poblacion):
        poblacion.append(copiar_red_neuronal(red_neuronal))
    return poblacion

#------------------------------------------------------------------------
def funcion_fitness(red_neuronal, X, Y):

    """Calcula el F1-score de la red neuronal en el conjunto de datos X. Y indica la clase que se predice"""
    # Colocar la red neuronal en modo de evaluación
    red_neuronal.eval()
    with torch.no_grad():
        # Ejecuta el modelo: forward pass
        y_pred = red_neuronal(X)
        # Calcula la clase predicha que contiene el mayor valor de la predicción
        y_pred = torch.argmax(y_pred, axis=1)
        # Calcula la métrica F1
        return f1_score(Y, y_pred, average="macro")  # Usar F1 macro

#------------------------------------------------------------------------
def evaluar_poblacion(poblacion, X, Y):
    """
    Evalua a todos los individuos de la población
    """
    fitness = []
    for individuo in poblacion:
        val_fitness = funcion_fitness(red_neuronal=individuo, X=X, Y=Y)
        fitness.append(val_fitness)
    return np.array(fitness)

#--------------------------------------------------------------------
def evaluar_poblacion_paralelo(poblacion,  X, Y, n_jobs=-1):
    """Evalúa la población en paralelo usando multiprocessing."""
    print(f"Evaluando población en paralelo ({n_jobs} núcleos)...")
    # Configurar el número de workers (n_jobs=-1 usa todos los núcleos)
    n_jobs = cpu_count() if n_jobs == -1 else n_jobs

    # Evaluar en paralelo

    resultados = Parallel(
    n_jobs = n_jobs-1,
    timeout = 300,  # 5min
    verbose = 10
    )(
        delayed(funcion_fitness)(red_neuronal=individuo, X=X, Y=Y)
        for individuo in poblacion
    )

    fitness = np.array(resultados)

    return fitness


#------------------------------------------------------------------------
def seleccionar_padres(poblacion, fitness, k=5):
    """Selección por torneo: el mejor de dos sobrevive."""
    """Selecciona dos padres usando selección por torneo."""
    torneo = random.sample(list(zip(poblacion, fitness)), k=k)
    # print("torneo:", torneo)
    torneo.sort(key=lambda x: x[1], reverse=True)  # Ordenar por F1-score
    return torneo[0][0], torneo[1][0]  # Retornar los dos mejores individuos

#------------------------------------------------------------------------
def elitismo(poblacion, fitness, tam_elite=2):
    """Selecciona los 'tam_elite' mejores best_individuos."""
    # Ordenar por fitness (mayor = mejor)
    rank_indices = np.argsort(fitness)[::-1]
    elites = [poblacion[i] for i in rank_indices[:tam_elite]]
    return elites


#------------------------------------------------------------------------
def cruzar_intercambio(padre1, padre2, prob_intercambio=0.7, prob_cruza = 0.9):
    """Cruce: intercambia pesos de dos padres de acuerdo a la probabilidad de intercambio."""
    # Si pasa el umbral de probabilidad de cruza se aplica el intercambio
    if np.random.rand() <  prob_cruza:
        # Crea el modelo de la red neuronal
        hijo1 = HumorRoBERTuito()
        hijo2 = HumorRoBERTuito()
        hijo1.load_state_dict(padre1.state_dict())
        hijo2.load_state_dict(padre2.state_dict())
        with torch.no_grad():
            # padre1.parameters() Obtiene los parámetros de la red: weight y bias
            for param_h1, param_h2,  param_p1, param_p2 in zip(hijo1.classifier.parameters(), hijo2.classifier.parameters(),  padre1.classifier.parameters(), padre2.classifier.parameters()):
                mask = torch.rand_like(param_h1) > prob_intercambio
                param_h1.copy_(mask * param_p1 + (~mask) * param_p2)
                param_h2.copy_(mask * param_p2 + (~mask) * param_p1)

    else:
        hijo1, hijo2 = padre1, padre2

    return hijo1, hijo2

#------------------------------------------------------------------------
def cruzar_aritmetica(padre1, padre2, alpha=0.5, prob_cruza = 0.9):

    # Si pasa el umbral de probabilidad de cruza se aplica el intercambio
    if np.random.rand() <  prob_cruza:
        """Cruce: Aplica la cruza aritmética"""
        # Crea el modelo de la red neuronal
        hijo1 = HumorRoBERTuito()
        hijo2 = HumorRoBERTuito()
        hijo1.load_state_dict(padre1.state_dict())
        hijo2.load_state_dict(padre2.state_dict())
        with torch.no_grad():
            # padre1.parameters() Obtiene los parámetros de la red: weight y bias
            for param_h1, param_h2,  param_p1, param_p2 in zip(hijo1.classifier.parameters(), hijo2.classifier.parameters(),  padre1.classifier.parameters(), padre2.classifier.parameters()):
                param_h1.copy_((alpha * param_p1) + ((1 - alpha) * param_p2))
                param_h2.copy_((alpha * param_p2) + ((1 - alpha) * param_p1))

    else:
        hijo1, hijo2 = padre1, padre2

    return hijo1, hijo2

#------------------------------------------------------------------------
def mutar(red_neuronal, intensidad_ruido=0.01, TASA_MUTACION = 0.1):
    """Mutación: añade ruido gaussiano a los pesos con probabilidad TASA_MUTACION e intensidad (intensidad_ruido)."""
    if np.random.rand() < TASA_MUTACION:
        with torch.no_grad():
            for param in red_neuronal.classifier.parameters():
                # Devuelve un tensor con el mismo tamaño que la entrada que se rellena con números aleatorios
                # de una distribución normal con media 0 y varianza 1.
                ruido = torch.randn_like(param) * intensidad_ruido
                param.add_(ruido)
    return red_neuronal

#------------------------------------------------------------------------
# --- Algoritmo Evolutivo ---
def algoritmo_evolutivo(modelo_red_neuronal, X, Y, tam_poblacion, tam_generaciones, run_paralelo=False):
    """Ejecuta el algoritmo evolutivo para optimizar la red neuronal."""
    poblacion = inicializar_poblacion(modelo_red_neuronal, tam_poblacion)

    for generacion in range(tam_generaciones):
        if run_paralelo:
            fitness = evaluar_poblacion_paralelo(poblacion, X, Y, n_jobs=-1)
        else:
            fitness = evaluar_poblacion(poblacion, X, Y)


        # Reporte de generación
        print(f"Generación {generacion} - Mejor precisión: {max(fitness):.4f}")

        nueva_poblacion = []
        for _ in range(tam_poblacion // 2):
            # Selección, cruce y mutación
            padre1, padre2 = seleccionar_padres(poblacion,fitness, k=5 )
            hijo1, hijo2 = cruzar_intercambio(padre1, padre2)
            nueva_poblacion.append(mutar(hijo1))
            nueva_poblacion.append(mutar(hijo2))

        #-----------------
        # Población: Los hijos sustituyen a los padres
        #-----------------
        # poblacion = nueva_poblacion

        #-----------------
        # Población con elitismo de padres y parte de los hijos
        #-----------------

        K_best_padres = 10
        poblacion[:K_best_padres] = elitismo(poblacion, fitness, K_best_padres)
        poblacion[K_best_padres:] = nueva_poblacion[K_best_padres: ]


    if run_paralelo:
        fitness = evaluar_poblacion_paralelo(poblacion, X, Y, n_jobs=-1)
    else:
        fitness = evaluar_poblacion(poblacion, X, Y)

    # Evaluar el mejor modelo en train
    best_individuo = poblacion[np.argmax(fitness)]
    test_f1 = funcion_fitness(best_individuo, X, Y)
    print(f"\nF1-score final en Train: {test_f1:.3f}")
    return best_individuo

In [ ]:
best_individuo = algoritmo_evolutivo(model, X_train, y_train, tam_poblacion=20, tam_generaciones=50, run_paralelo=False)

Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Cargando RoBERTuito...


...

Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Cargando RoBERTuito...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

TypeError: HumorRoBERTuito.forward() missing 1 required positional argument: 'attention_mask'

In [ ]:
print("\nCargando dataset de prueba...")

dataset_test = pd.read_json(
    "./Datasets/dataset_humor_test.json",
    lines=True
)

X_test = dataset_test["text"].values

# =========================================================
# TEST DATASET
# =========================================================

test_dataset = HumorTextDataset(
    X_test,
    labels=None,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# =========================================================
# TEST DATALOADER
# =========================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

# =========================================================
# INFERENCE
# =========================================================

best_red_neuronal = best_individuo

best_red_neuronal.eval()

predictions_list = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        predictions_list.extend(
            preds.cpu().numpy()
        )

y_pred_final = np.array(
    predictions_list
)

# =========================================================
# RESULTADOS
# =========================================================

print("\nPredicciones generadas.")

unique, counts = np.unique(
    y_pred_final,
    return_counts=True
)

print(
    "Conteo de clases:",
    dict(zip(unique, counts))
)

print(
    "\nPrimeras predicciones:"
)

print(
    y_pred_final[:20]
)